### Mashiur Rahman

# Project 2 — Preparing Stock Data for Machine Learning

## Part 1 — Clean and Prepare the Data

### Import Required Library and Load Dataset

In [1]:
import pandas as pd

stocks = pd.read_csv("historical_stocks.csv")
prices = pd.read_csv("historical_stock_prices.csv")

print("Stock information:", stocks.shape)
print("Daily prices:", prices.shape)

Stock information: (6460, 5)
Daily prices: (20973889, 8)


#### convert the date and sort

In [2]:
prices["date"] = pd.to_datetime(prices["date"])
prices.sort_values(["ticker", "date"], inplace=True)

print("Date type:", prices["date"].dtype)
display(prices[["ticker", "date", "close"]].head())

Date type: datetime64[us]


,ticker,date,close
13766141,A,1999-11-18,31.473534
13766149,A,1999-11-19,28.880543
13766150,A,1999-11-22,31.473534
13766151,A,1999-11-23,28.612303
13766152,A,1999-11-24,29.372318


#### remove the accidental stock record

In [3]:
print("Accidental record:")
display(stocks[stocks["ticker"] == "Symbol"])

stocks = stocks[stocks["ticker"] != "Symbol"].copy()

print("Stock rows after cleaning:", len(stocks))

Accidental record:


,ticker,exchange,name,sector,industry
74,Symbol,NYSE,NAME,SECTOR,INDUSTRY


Stock rows after cleaning: 6459


#### Review Unusual Prices

Project 1 found very high closing prices for SCON and extremely high adjusted closing prices for AAN. I will display these records again before deciding what to do with them.

#### inspect the unusual records

In [4]:
print("Highest closing prices:")
display(prices.nlargest(5, "close")[
    ["ticker", "date", "close", "adj_close", "volume"]
])

print("Highest adjusted closing prices:")
display(prices.nlargest(5, "adj_close")[
    ["ticker", "date", "close", "adj_close", "volume"]
])

Highest closing prices:


,ticker,date,close,adj_close,volume
12146443,SCON,2000-02-28,1779750.0,1779750.0,400
12146459,SCON,2000-02-29,1595250.0,1595250.0,400
12146546,SCON,2000-03-13,1363500.0,1363500.0,300
17889354,CODA,2001-02-05,1354500.0,1354500.0,34
2444619,TVIX,2013-02-26,1347500.0,1347500.0,100


Highest adjusted closing prices:


,ticker,date,close,adj_close,volume
6174033,AAN,1987-02-27,1.296296,1.894962e+19,346900
6174039,AAN,1987-03-10,1.296296,1.894962e+19,20200
6174040,AAN,1987-03-11,1.296296,1.894962e+19,18900
6174041,AAN,1987-03-12,1.296296,1.894962e+19,27000
6174044,AAN,1987-03-17,1.296296,1.894962e+19,31000


### Part 1 Cleaning Decision

I removed the accidental `Symbol` row because it contains column headings instead of stock information.

I kept the unusually high price records for now and flagged them for further investigation. In Project 1, SCON had a closing price of 1,779,750 on February 28, 2000. AAN had adjusted closing prices near 1.89 × 10¹⁹ in 1987, while its closing prices in those records were near 1.3. These AAN values are especially suspicious, but I cannot confirm their cause from the dataset alone. I would check the original source and the price-adjustment method before correcting or deleting them.

I also kept records with missing sector or industry information. Missing classifications alone do not prove that a stock record is incorrect.

## Part 2 — Create Useful Features

#### price change, daily return, and previous close:

In [5]:
by_stock = prices.groupby("ticker")

prices["close_lag_1"] = by_stock["close"].shift(1)
prices["price_change"] = prices["close"] - prices["close_lag_1"]
prices["daily_return"] = prices["price_change"] / prices["close_lag_1"]

#### moving averages:

In [6]:
prices["ma_5"] = by_stock["close"].transform(
    lambda values: values.rolling(5).mean()
)

prices["ma_20"] = by_stock["close"].transform(
    lambda values: values.rolling(20).mean()
)

#### volatility and  additional feature:

In [7]:
prices["volatility_20"] = prices.groupby("ticker")["daily_return"].transform(
    lambda values: values.rolling(20).std()
)

prices["high_low_range"] = prices["high"] - prices["low"]

#### check the new columns using AHH:

In [8]:
new_columns = [
    "ticker", "date", "close", "price_change", "daily_return",
    "close_lag_1", "ma_5", "ma_20", "volatility_20",
    "high_low_range"
]

display(prices.loc[prices["ticker"] == "AHH", new_columns].head(22))

,ticker,date,close,price_change,daily_return,close_lag_1,ma_5,ma_20,volatility_20,high_low_range
0,AHH,2013-05-08,11.58,NaN,NaN,NaN,NaN,NaN,NaN,0.430000
1,AHH,2013-05-09,11.55,-0.030000,-0.002591,11.58,NaN,NaN,NaN,0.160000
2,AHH,2013-05-10,11.60,0.050000,0.004329,11.55,NaN,NaN,NaN,0.100000
3,AHH,2013-05-13,11.65,0.049999,0.004310,11.60,NaN,NaN,NaN,0.099999
4,AHH,2013-05-14,11.53,-0.120000,-0.010300,11.65,11.582,NaN,NaN,0.100000
5,AHH,2013-05-15,11.60,0.070001,0.006071,11.53,11.586,NaN,NaN,0.060000
6,AHH,2013-05-16,11.74,0.139999,0.012069,11.60,11.624,NaN,NaN,0.200000
7,AHH,2013-05-17,11.76,0.020000,0.001704,11.74,11.656,NaN,NaN,0.150001
8,AHH,2013-05-20,11.73,-0.030001,-0.002551,11.76,11.672,NaN,NaN,0.200000
9,AHH,2013-05-21,11.83,0.100000,0.008525,11.73,11.732,NaN,NaN,0.230000


### Part 2 Answers

1. **What does `daily_return` tell us?**  
   It tells me the percentage change in a stock's closing price from its previous trading record. A positive value means the price rose; a negative value means it fell.

2. **What does a moving average tell us about a stock?**  
   It shows the average closing price over recent trading records. It helps me see the price trend without focusing on every daily change.

3. **What does high volatility indicate?**  
   High volatility means daily returns have varied a lot over the recent 20 trading records. The stock's returns have been less steady during that period.

4. **Why do lag and rolling features create missing values at the beginning of each stock's history?**  
   The first record has no earlier closing price. A 5-record average needs five prices, a 20-record average needs twenty prices, and 20-day volatility needs twenty daily returns. Until enough earlier records exist, those values cannot be calculated.

**Additional feature:** I created `high_low_range` by subtracting the day's low price from its high price. It gives a simple measure of movement within that trading day.

## Part 3 — Combine and Organize the Data

#### merge and check the number of rows:

In [9]:
rows_before = len(prices)

prices = prices.merge(
    stocks[["ticker", "exchange", "sector", "industry"]],
    on="ticker",
    how="left",
    validate="many_to_one",
    sort=False
)

print("Price rows before merge:", rows_before)
print("Rows after merge:", len(prices))
print("Price rows added by mistake:", len(prices) - rows_before)
print("Rows with no matching exchange:", prices["exchange"].isna().sum())

Price rows before merge: 20973889
Rows after merge: 20973889
Price rows added by mistake: 0
Rows with no matching exchange: 0


#### check duplicate ticker and date pairs:

In [10]:
print(
    "Duplicate ticker and date pairs:",
    prices.duplicated(subset=["ticker", "date"]).sum()
)

Duplicate ticker and date pairs: 0


### Missing Values Created by Features

I will remove rows that do not yet have enough earlier trading records to calculate the lag and rolling features. I will keep missing stock classifications by labeling them "Unknown".

#### handle missing values:

In [11]:
feature_columns = [
    "close_lag_1", "price_change", "daily_return",
    "ma_5", "ma_20", "volatility_20"
]

print("Missing feature values before cleaning:")
display(prices[feature_columns].isna().sum())

rows_before = len(prices)
prices.dropna(subset=feature_columns, inplace=True)

print("Rows removed:", rows_before - len(prices))
print("Rows remaining:", len(prices))

prices["exchange"] = prices["exchange"].fillna("Unknown")
prices["sector"] = prices["sector"].fillna("Unknown")
prices["industry"] = prices["industry"].fillna("Unknown")

Missing feature values before cleaning:


close_lag_1        5685
price_change       5685
daily_return       5685
ma_5              22735
ma_20            107884
volatility_20    113555
dtype: int64

Rows removed: 113555
Rows remaining: 20860334


#### one-hot encode exchange and sector:

In [12]:
category_columns = pd.get_dummies(
    prices[["exchange", "sector"]],
    dtype="int8"
)

print("New category columns:")
print(category_columns.columns.tolist())

display(category_columns.head())

New category columns:
['exchange_NASDAQ', 'exchange_NYSE', 'sector_BASIC INDUSTRIES', 'sector_CAPITAL GOODS', 'sector_CONSUMER DURABLES', 'sector_CONSUMER NON-DURABLES', 'sector_CONSUMER SERVICES', 'sector_ENERGY', 'sector_FINANCE', 'sector_HEALTH CARE', 'sector_MISCELLANEOUS', 'sector_PUBLIC UTILITIES', 'sector_TECHNOLOGY', 'sector_TRANSPORTATION', 'sector_Unknown']


,exchange_NASDAQ,exchange_NYSE,sector_BASIC INDUSTRIES,sector_CAPITAL GOODS,sector_CONSUMER DURABLES,sector_CONSUMER NON-DURABLES,sector_CONSUMER SERVICES,sector_ENERGY,sector_FINANCE,sector_HEALTH CARE,sector_MISCELLANEOUS,sector_PUBLIC UTILITIES,sector_TECHNOLOGY,sector_TRANSPORTATION,sector_Unknown
20,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0
21,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0
22,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0
23,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0
24,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0


### Part 3 Answers and Decisions

**Merge:** I joined the price records to the cleaned stock information using `ticker`. I checked the row counts and duplicate `ticker` and `date` pairs after the merge.

**Missing feature values:** I removed the early records for each ticker that did not have enough earlier data for all lag and rolling features. These missing values were expected, so I did not replace them with zero.

**Missing categories:** I labeled missing exchange, sector, or industry information as `Unknown`. This keeps the remaining price records without pretending to know their classifications.

**Why use one-hot encoding?** One-hot encoding gives each exchange or sector its own 0-or-1 column. Assigning NASDAQ = 1 and NYSE = 2 might wrongly suggest that NYSE is numerically greater than NASDAQ.

## Part 4 — Prepare the Final Dataset

I will keep the stock identifier, date, daily prices, volume, the features I created, and the stock information. I will also keep the one-hot encoded exchange and sector columns.

I will leave `adj_close` out of the final dataset because Project 1 found extremely large adjusted values whose cause has not been verified. I will keep two simple flags so these unusual records can be found again. The flags identify values for review; they do not prove an error.

#### 4.1 Select the Final Features

In [13]:
# Mark unusually large values found during Project 1.
prices["high_close_flag"] = prices["close"] > 1_000_000
prices["high_adj_close_flag"] = prices["adj_close"] > 1_000_000_000_000

# Add the one-hot encoded columns from Part 3.
encoded_names = category_columns.columns.tolist()
prices[encoded_names] = category_columns

selected_columns = [
    "ticker", "date",
    "open", "high", "low", "close", "volume",
    "price_change", "daily_return", "close_lag_1",
    "ma_5", "ma_20", "volatility_20", "high_low_range",
    "exchange", "sector", "industry",
    "high_close_flag", "high_adj_close_flag"
] + encoded_names

# Remove columns that were not selected without making another
# full copy of this large dataset.
prices.drop(columns=prices.columns.difference(selected_columns), inplace=True)

final_data = prices
del category_columns

print("Final dataset shape:", final_data.shape)
display(final_data.head())

Final dataset shape: (20860334, 34)


,ticker,open,close,low,high,volume,date,close_lag_1,price_change,daily_return,...,sector_CONSUMER NON-DURABLES,sector_CONSUMER SERVICES,sector_ENERGY,sector_FINANCE,sector_HEALTH CARE,sector_MISCELLANEOUS,sector_PUBLIC UTILITIES,sector_TECHNOLOGY,sector_TRANSPORTATION,sector_Unknown
20,A,33.172390,32.859444,32.501789,33.708870,3708000,1999-12-17,33.798283,-0.938839,-0.027778,...,0,0,0,0,0,0,0,0,0,0
21,A,33.082977,33.530045,32.993561,33.574749,1196800,1999-12-20,32.859444,0.670601,0.020408,...,0,0,0,0,0,0,0,0,0,0
22,A,33.395924,33.351215,32.904148,33.395924,2259400,1999-12-21,33.530045,-0.178829,-0.005333,...,0,0,0,0,0,0,0,0,0,0
23,A,33.351215,34.021816,33.127682,34.021816,1905700,1999-12-22,33.351215,0.670601,0.020107,...,0,0,0,0,0,0,0,0,0,0
24,A,33.977112,35.586552,33.932404,35.765381,2159400,1999-12-23,34.021816,1.564735,0.045992,...,0,0,0,0,0,0,0,0,0,0


#### 4.2 Check the Final Dataset

In [14]:
print("Rows and columns:", final_data.shape)
print("\nColumn types:")
print(final_data.dtypes)

print("\nMissing values:")
print(final_data.isna().sum())

print(
    "\nDuplicate ticker and date pairs:",
    final_data.duplicated(subset=["ticker", "date"]).sum()
)

print("\nEarliest date:", final_data["date"].min())
print("Latest date:", final_data["date"].max())

print("\nHigh close flags:", final_data["high_close_flag"].sum())
print("High adjusted close flags:", final_data["high_adj_close_flag"].sum())

Rows and columns: (20860334, 34)

Column types:
ticker                                     str
open                                   float64
close                                  float64
low                                    float64
high                                   float64
volume                                   int64
date                            datetime64[us]
close_lag_1                            float64
price_change                           float64
daily_return                           float64
ma_5                                   float64
ma_20                                  float64
volatility_20                          float64
high_low_range                         float64
exchange                                   str
sector                                     str
industry                                   str
high_close_flag                           bool
high_adj_close_flag                       bool
exchange_NASDAQ                           int8
exchange_NYS

#### save the required CSV:

In [15]:
final_data.to_csv("stock_prepared.csv", index=False)
print("Saved stock_prepared.csv")

Saved stock_prepared.csv


### Part 4 Final Dataset

I kept ticker and `date` so each daily record can be identified. I kept the open, high, low, close, and volume columns because they describe that day's trading. The new price-change, return, lag, moving-average, volatility, and high-low-range columns describe recent stock behaviour.

I included exchange, sector, and industry for context, along with numerical one-hot columns for exchange and sector. I did not include `adj_close` because its extreme AAN values need further investigation. The two flag columns make unusually large original close and adjusted-close values easy to find later. I removed early records where the lag or rolling features could not yet be calculated, then checked the final dataset before saving it as `stock_prepared.csv`.

## Part 5 — Documentation and Findings

### 1. What data-quality issues did I investigate?

I checked the stock information and price data for missing values, duplicate ticker and date records, and unusual prices. The stock information file had one row containing column headings instead of stock data. I removed that row, leaving 6,459 stock information rows. I also found missing sector and industry classifications. The price data had unusually high closing prices for SCON and adjusted closing prices for AAN.

### 2. What decisions did I make about unusual observations?

I kept the unusual price records because I could not confirm from these files alone whether they were errors. I added flags to make them easy to review later. In the final dataset, 1 record has the high closing-price flag and 448 records have the high adjusted-closing-price flag. I excluded `adj_close` from the final columns because its extreme AAN values need investigation. I changed missing stock classifications to `Unknown`.

I also removed 113,555 early price records that did not have enough previous trading records to calculate all the lag and rolling features. I did not fill those missing calculations with zero.

### 3. What new features did I create?

I created the previous closing price (`close_lag_1`), daily price change (`price_change`), daily return (`daily_return`), 5-day and 20-day moving averages (`ma_5` and `ma_20`), 20-day volatility (`volatility_20`), and the difference between the day's high and low prices (`high_low_range`). I also created 0-or-1 columns for exchange and sector, plus two unusual-price flags.

### 4. Which feature may be useful for future analysis, and why?

I think `daily_return` may be useful because it measures a price change relative to the previous closing price. This makes it easier to compare price movements between stocks with different prices.

### 5. What does my final prepared dataset contain?

The final dataset has **20,860,334 rows and 34 columns**. It contains ticker and date, selected daily prices and volume, the new features, exchange, sector, industry, encoded exchange and sector columns, and unusual-price flags. Its dates run from **1970-01-30 to 2018-08-24**. My final checks found **no missing values** and **no duplicate ticker and date combinations**.

### 6. What additional information or features might help a predictive model?

Market index returns, company financial information, and interest rates could add useful context. I could also calculate longer-term returns or average trading volume. When building a model to predict a future price, I would use only information that was available before the date being predicted.